In [6]:
print("Ok")

Ok


In [7]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [8]:
# Extract data from PDF files
def load_pdf_file(data):
    loader = DirectoryLoader(
        data,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    documents = loader.load()
    return documents

In [10]:
extracted_data = load_pdf_file(data="../Data/")

In [11]:
def text_splitter(documents):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20,
    )
    split_documents = text_splitter.split_documents(documents)
    return split_documents

text_split_documents = text_splitter(extracted_data)
print(f"Number of documents loaded: {len(extracted_data)}")
print(f"Number of split documents: {len(text_split_documents)}")

Number of documents loaded: 759
Number of split documents: 7024


In [12]:
import langchain_huggingface


def download_hugging_face_embeddings():
        embeddings=langchain_huggingface.HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
        return embeddings



In [13]:
embeddings = download_hugging_face_embeddings()

c:\Users\melke\Desktop\Medical chatbot\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4961.05it/s]


In [ ]:
cd "c:\\Users\\melke\\Desktop\\Medical chatbot"

c:\Users\melke\Desktop\Medical chatbot


c:\Users\melke\Desktop\Medical chatbot\.venv\lib\site-packages\IPython\core\magics\osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [14]:
query = "What is the purpose of this clinical trial?"
print("len",len(embeddings.embed_query(query)))

len 384


In [15]:

import os

from dotenv import load_dotenv
load_dotenv()


True

In [16]:
PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

In [ ]:
from pinecone import Pinecone, ServerlessSpec
import os

pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

index_name = "medical-chatbot"

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

In [41]:
import os
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [20]:
#Embed each chunk and upsert the embeddings into your Pinecone index.
from langchain_pinecone import PineconeVectorStore

index_name = "medical-chatbot"

docsearch = PineconeVectorStore.from_documents(
        documents=text_split_documents,
        index_name=index_name,
        embedding=embeddings,
)

In [21]:
# Load Existing index
from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [22]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [81]:
retrieve_query = retriever.invoke("What is dermatitis?")
print("Retrieved query:", retrieve_query)

Retrieved query: [Document(id='2a58d524-7cf4-4cd0-96ca-5da5f2b1d3cf', metadata={'creationdate': '2004-12-18T17:16:32-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:35:04-06:00', 'page': 423.0, 'page_label': '424', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': '..\\Data\\Gale Encyclopedia of Medicine Vol. 2 (C-F).pdf', 'total_pages': 759.0}, page_content='National Institute of Mental Health. Mental Health Public\nInquiries, 5600 Fishers Lane, Room 15C-05, Rockville,\nMD 20857. (888) 826-9438. <http://www.nimh.nih.gov>.\nPaula Anne Ford-Martin\nDermabrasion see Skin resurfacing\nDermatitis\nDefinition\nDermatitis is a general term used to describe inflam-\nmation of the skin.\nDescription\nMost types of dermatitis are characterized by an\nitchy pink or red rash.\nContact dermatitis is an allergic reaction to some-'), Document(id='218c96ec-1303-4168-ba61-475c462f3049', metadata={'creationdate': '2004-12-18T17:16:32-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:35:04-06:

In [74]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3:latest",
    temperature=0.4,
    num_predict=500
)

In [76]:


from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise.\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])




In [82]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [84]:

response = rag_chain.invoke({"input": "What is dermatitis?"})
print(response["answer"])

According to the provided context, dermatitis is a general term used to describe inflammation of the skin.
